# Проверка сетки детектора и обучение головы на train + val

Данные соревнования, прежний приватный clevr_verifier, две T4. Первый этап сравнивает модели на val, второй готовит финальный CSV.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, display
from scipy.special import expit
from sklearn.metrics import roc_auc_score

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
assert torch.cuda.device_count() == 2, "Выберите GPU T4 x2"
ROOT = Path(tempfile.mkdtemp(prefix="clevr_verify_"))
RESULT = ROOT / "verify_result"
RESULT.mkdir()
print("Рабочая папка", ROOT, flush=True)

def unpack(path, dest):
    with zipfile.ZipFile(path) as z:
        for name in z.namelist():
            if not (dest / name).resolve().is_relative_to(dest.resolve()):
                raise ValueError(name)
        z.extractall(dest)

def find_data(parent):
    return [p.parent for p in parent.rglob("train.csv")
            if all((p.parent / n).is_file() for n in ("val.csv","test.csv","sample_submission.csv"))
            and all((p.parent / n).is_dir() for n in ("train","val","test"))]

locations = list(INPUT.rglob("verifier_manifest.json"))
if not locations:
    archives = list(INPUT.rglob("clevr_verifier.zip"))
    if len(archives) != 1:
        raise ValueError("Подключите один Private Dataset из clevr_verifier.zip")
    unpack(archives[0], ROOT / "package")
    locations = [ROOT / "package/verifier_manifest.json"]
if len(locations) != 1:
    raise ValueError("Подключено несколько версий весов")
package = locations[0].parent
for key, digest in json.loads(locations[0].read_text()).items():
    with (package / key).open("rb") as f:
        if hashlib.file_digest(f, "sha256").hexdigest() != digest:
            raise ValueError(key)
datasets = find_data(INPUT)
if not datasets:
    for j, path in enumerate(INPUT.rglob("*.zip")):
        with zipfile.ZipFile(path) as z:
            names = {Path(k).name for k in z.namelist()}
        if {"train.csv","val.csv","test.csv","sample_submission.csv"} <= names:
            unpack(path, ROOT / "inputs" / str(j))
    datasets = find_data(ROOT / "inputs")
if len(datasets) != 1:
    raise ValueError("Подключите данные соревнования")
DATA = datasets[0]
env = dict(os.environ, CLEVR_DATA=str(DATA), CLEVR_PRIORS=str(package/"priors"),
           CLEVR_WEIGHTS=str(package/"weights_in"), PYTHONPATH=str(package),
           PYTHONUNBUFFERED="1", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2")
WORKER = [sys.executable, "-m", "scripts.research.occlusion_verifier"]
print(DATA, *[torch.cuda.get_device_name(i) for i in range(2)], sep=" | ", flush=True)

def run(gpu, *args):
    stage = "_".join(args).replace("--", "")
    done = ROOT / (stage + ".done")
    if done.exists():
        return
    logfile = RESULT / (stage + ".log")
    with logfile.open("a") as f, subprocess.Popen([*WORKER, *args], cwd=ROOT,
            env={**env, "CUDA_VISIBLE_DEVICES":str(gpu)}, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as job:
        for line in job.stdout:
            f.write(line)
            print(f"[{gpu}] {line}", end="", flush=True)
        if job.wait():
            shutil.copy2(logfile, WORK / "verifier_failed.log")
            raise RuntimeError(f"{stage} завершился с ошибкой; повторите эту ячейку")
    done.touch()

def both(left, right):
    with ThreadPoolExecutor(2) as pool:
        jobs = [pool.submit(run, gpu, *args) for gpu, args in enumerate((left, right))]
        for job in jobs:
            job.result()


worker_path = ROOT/'phase_refit.py'
worker_path.write_text('"""Compare detector grid phases at inference and refit the union head on train + val."""\nimport argparse\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.nn import functional as F\n\nfrom scripts.research import occlusion_verifier as v\n\n\n@torch.inference_mode()\ndef extract(split, shard, device, single=False):\n    names = pd.read_csv(v.DATA / f"{split}.csv").image.unique()[shard::2]\n    models = [v.ResNetCenters(full=True), v.CenterDetector()]\n    heads = []\n    for model, key in zip(models, ("r50_img","thaw")):\n        model.load_state_dict(v.load(v.PRIORS / f"{key}.pt"))\n        model.to(device, memory_format=torch.channels_last).eval()\n        heads.append(v.model_head(key, device))\n    matrix = v.projection().to(device)\n    phases = [(0,0)] if split == "train" or single else [(0,0),(2,0),(0,2),(2,2)]\n    parts = {s:[] for s in phases}\n    with ThreadPoolExecutor(3) as pool:\n        for start in range(0,len(names),128):\n            images = np.stack(list(pool.map(v.read_rgb,[v.DATA/split/n for n in names[start:start+128]])))\n            rgb = torch.from_numpy(images).to(device).permute(0,3,1,2).float().div_(255)\n            native = v.detector_outputs(models[1],heads[1],rgb,matrix)\n            for dx,dy in phases:\n                img = F.pad(rgb,(dx,0,dy,0),mode="replicate")[:,:,:v.HEIGHT,:v.WIDTH] if dx or dy else rgb\n                r50 = v.detector_outputs(models[0],heads[0],img)\n                shift = np.array([dx/v.WIDTH,dy/v.HEIGHT])\n                r50[:,:,-6:-4] -= shift\n                r50[:,:,-2:] -= 2*shift\n                rows = [v.merge(a,b) for a,b in zip(r50,native)]\n                parts[dx,dy].append(tuple(map(np.stack,zip(*rows))))\n    for (dx,dy), stacks in parts.items():\n        x,xy,valid = (np.concatenate(col) for col in zip(*stacks))\n        np.savez(v.CACHE/f"{split}_{dx}{dy}_{shard}.npz",names=names.astype(str),x=x,xy=xy,valid=valid)\n\n\ndef assemble(split,phase):\n    tables = v.CACHE/"tables"\n    tables.mkdir(exist_ok=True)\n    with np.load(v.CACHE/f"{split}_{phase}_0.npz") as a,np.load(v.CACHE/f"{split}_{phase}_1.npz") as b:\n        merged = {}\n        for key in a.files:\n            x,y=a[key],b[key]\n            dest=np.empty((len(x)+len(y),*x.shape[1:]),dtype=np.result_type(x.dtype,y.dtype))\n            dest[::2],dest[1::2]=x,y\n            merged[key]=dest\n        np.savez(tables/f"{split}_{phase}.npz",**merged)\n\n\nif __name__=="__main__":\n    p=argparse.ArgumentParser()\n    p.add_argument("action",choices=("extract","train","infer"))\n    p.add_argument("split")\n    p.add_argument("number",type=int)\n    p.add_argument("phase",default="00",nargs="?")\n    p.add_argument("fit",default="train",nargs="?")\n    args=p.parse_args()\n    _,device=v.setup(args.number)\n    v.CACHE.mkdir(exist_ok=True)\n    if args.action=="extract":\n        extract(args.split,args.number,device,args.phase==\'single\')\n    else:\n        v.OUT=Path("verify_result")/args.fit\n        v.OUT.mkdir(parents=True,exist_ok=True)\n        if args.action=="train":\n            # SceneData reads the train.csv and table under this isolated training root.\n            v.DATA=Path("fit")/args.fit\n            v.CACHE=v.DATA\n            v.train("control",args.number,12,device)\n        else:\n            v.DATA=Path("phase")/args.phase\n            v.CACHE=v.DATA\n            v.infer(args.split,"control",args.number,device)\n            path=v.OUT/f"control_{args.number}_{args.split}.npz"\n            path.replace(v.OUT/f"{args.number}_{args.split}_{args.phase}.npz")\n',encoding='utf8')
WORKER = [sys.executable,str(worker_path)]
sys.path[:0]=[str(ROOT),str(package)]

In [ ]:
from phase_refit import assemble, v
os.chdir(ROOT)
seeds = (421,853)
phases = ("00","20","02","22")
def prepare(split, phases):
    both(*[("extract",split,str(i),"single" if len(phases)==1 else "all") for i in (0,1)])
    for phase in phases:
        assemble(split,phase)
        folder = ROOT/"phase"/phase
        (folder/"tables").mkdir(exist_ok=True,parents=True)
        shutil.copyfile(DATA/f"{split}.csv",folder/f"{split}.csv")
        shutil.copyfile(ROOT/"verify_cache"/"tables"/f"{split}_{phase}.npz",folder/"tables"/f"{split}.npz")

def fit(name, splits):
    folder = ROOT/"fit"/name
    (folder/"tables").mkdir(exist_ok=True,parents=True)
    frames = [pd.read_csv(DATA/f"{s}.csv") for s in splits]
    frames_all = pd.concat(frames,ignore_index=True)
    assert frames_all.id.is_unique
    frames_all.to_csv(folder/"train.csv",index=False)
    parts=[]
    for s in splits:
        with np.load(ROOT/"phase/00/tables"/f"{s}.npz") as a:
            parts.append(dict(a))
    merged={k:np.concatenate([a[k] for a in parts]) for k in parts[0]}
    assert np.array_equal(frames_all.image.unique(),merged["names"])
    np.savez(folder/"tables/train.npz",**merged)
    both(*[("train","train",str(seed),"00",name) for seed in seeds])

def predict(split,fit_name,views):
    frame=pd.read_csv(DATA/f"{split}.csv")
    per_view=[]
    for phase in views:
        both(*[("infer",split,str(seed),phase,fit_name) for seed in seeds])
        paired=[]
        for seed in seeds:
            # Keep predictions for each grid phase, including after re-running a cell.
            path=RESULT/fit_name/f"control_{seed}_{split}.npz"
            saved=path.with_name(f"{seed}_{split}_{phase}.npz")
            with np.load(saved) as a:
                assert np.array_equal(a["ids"],frame.id)
                paired.append(a["score"])
        per_view.append(np.mean(paired,axis=0))
    with np.load(package/"priors"/f"occ_{split}.npz") as a:
        assert np.array_equal(a["ids"],frame.id)
        anchor=a["score"]
    def blend(union):
        easy=~frame.category.isin(["counting","spatial_counting","comparison"]) & (anchor*union>0)
        return (.75*anchor+.25*union)*np.where(easy,1.5,1.)
    return frame,blend(per_view[0]),blend(np.mean(per_view,axis=0))

prepare("train",("00",))
prepare("val",phases)
fit("train",["train"])
frame,base,multi=predict("val","train",phases)
y=frame.label.to_numpy()
auc=lambda z: roc_auc_score(y,z)
print("Val: single / four",auc(base),auc(multi))
print("Ошибки и ничьи",[(int(((z<0)&(y==1)).sum()),int(((z==0)&(y==1)).sum())) for z in (base,multi)])
scene=pd.factorize(frame.image)[0]
rng=np.random.default_rng(765)
delta=[]
for _ in range(800):
    n=scene.max()+1
    w=rng.multinomial(n,np.full(n,1/n))[scene]
    delta.append(roc_auc_score(y,multi,sample_weight=w)-roc_auc_score(y,base,sample_weight=w))
ci=np.quantile(delta,[.025,.975])
print("95% интервал прироста",ci)
use_multi=ci[0]>0
np.savez(RESULT/"validation.npz",ids=frame.id,base=base,multi=multi,interval=ci)

In [ ]:
fit("trainval",["train","val"])
views=phases if use_multi else ("00",)
prepare("test",views)
frame,base,multi=predict("test","trainval",views)
sample=pd.read_csv(DATA/"sample_submission.csv")
assert sample.id.is_unique and frame.id.is_unique and set(frame.id)==set(sample.id)
for name,values in ([("union_tv",base),("phase_tv",multi)] if use_multi else [("union_tv",base)]):
    sample["label"]=pd.Series(expit(values),index=frame.id).loc[sample.id].to_numpy()
    path=WORK/f"{name}.csv"
    sample.to_csv(path,index=False,float_format="%.17g")
    assert sample.equals(pd.read_csv(path,float_precision="round_trip")) and sample.label.between(0,1).all()
shutil.make_archive(str(WORK/"phase_results"),"zip",RESULT)
os.chdir(WORK)
for name in ["union_tv.csv","phase_results.zip"]+(["phase_tv.csv"] if use_multi else []):
    display(FileLink(name))